# NestedSimPy — Event Latency

[NestedSimPy](https://nestedsimpy.github.io/) adapts SimPy's official [Event Latency example](https://simpy.readthedocs.io/en/latest/examples/latency.html) — a delayed message channel built on a `Store` — so the outer behavior is unchanged while inner simulations fork at each triggering event. This example uses `NestedStore`.

See the [example page](https://nestedsimpy.github.io/official-parity/event-latency.html) for the side-by-side plain/nested code.

## 1. Install

_Pre-release: NestedSimPy installs from a hosted wheel. (After the public release this becomes `pip install nestedsimpy`.)_

In [ ]:
# Pre-release install from a hosted wheel (Google Drive).
!pip install -q gdown
import gdown
gdown.download(id="1N7mlgDVpVids6Ekr4p2e-gEUrUodiEuq",
               output="nestedsimpy-0.1.0-py3-none-any.whl", quiet=True)
!pip install -q "nestedsimpy-0.1.0-py3-none-any.whl[plot]"

import nestedsimpy
print("NestedSimPy ready —", len(nestedsimpy.__all__), "public objects")

## 2. Run the nested example

The model is written to a file and run as a subprocess; the output below is the **outer** trajectory and matches the plain SimPy example.

In [ ]:
%%writefile event_latency_colab.py
# --- inline prelude (replaces the examples' local _imports shim) ---
import argparse, os, random, shutil, sys, itertools
from pathlib import Path

import simpy
import nestedsimpy
from nestedsimpy import (
    NestedEnvironment, NestedResource, NestedPreemptiveResource,
    NestedStore, NestedContainer,
)
try:
    from nestedsimpy.postprocess import (
        package_latest_run, relocate_raw_artifacts, export_realizations,
    )
except Exception:  # pragma: no cover
    package_latest_run = relocate_raw_artifacts = export_realizations = None

DEFAULT_OUT_ROOT = Path("nested_output")
DEFAULT_AUTOPLOT = False
REPO_ROOT = Path(".")
PACKAGE_ROOT = Path(".")

def default_out(*parts):
    p = DEFAULT_OUT_ROOT.joinpath(*map(str, parts)); p.mkdir(parents=True, exist_ok=True); return p

def set_nested_output_folder(*parts):
    p = Path(os.path.join(*[str(x) for x in parts])); p.mkdir(parents=True, exist_ok=True); return p

def should_emit_user_output(env=None):
    if env is None:
        return True
    f = getattr(env, "should_emit_user_output", None)
    return bool(f()) if callable(f) else getattr(env, "_ns_run_kind", "outer") != "inner"

def user_print(*args, env=None, **kw):
    if should_emit_user_output(env):
        print(*args, **kw)
# --- end prelude ---

"""
Event Latency example (nested-sim adaptation).

Covers:

- Resources: Store

Scenario:
  This example shows how to separate the time delay of events between
  processes from the processes themselves.

When Useful:
  When modeling physical things such as cables, RF propagation, etc.  it
  better encapsulation to keep this propagation mechanism outside of the
  sending and receiving processes.

  Can also be used to interconnect processes sending messages

Example by:
  Keith Smith
"""

import itertools


SIM_DURATION = 100
NESTED_OUTPUT_FOLDER = set_nested_output_folder("simpy_examples", "event_latency")


class Cable:
    """This class represents the propagation through a cable."""

    def __init__(self, env, delay):
        self.env = env
        self.delay = delay
        self.store = env.register(
            NestedStore(env, capacity=float("inf"), nested_id="cable_store")
        )

    def latency(self, value):
        """Delay ``value`` before placing it into the downstream store."""

        yield self.env.nested_timeout(
            {"distribution": "deterministic", "value": self.delay}, label="cable_delay"
        )
        yield self.store.put(value)

    def put(self, value):
        """Public API: schedule a value to traverse the cable."""
        self.env.process(self.latency(value))

    def get(self):
        """Public API: retrieve the next delivered value."""
        return self.store.get()


def sender(env, cable):
    """A process which randomly generates messages."""
    msg_id = itertools.count()
    while True:
        yield env.nested_timeout(
            {"distribution": "deterministic", "value": 5}, label="send_interval"
        )
        cid = next(msg_id)
        cable.put({"item_id": cid, "msg": f"Sender sent this at {env.now:g}"})


def receiver(env, cable):
    """A process which consumes messages."""
    while True:
        msg = yield cable.get()
        text = msg.get("msg") if isinstance(msg, dict) else msg
        user_print(f"Received this at {env.now:g} while {text}", env=env)


def main():
    """Run the nested-sim event latency example and package the outputs.

    Returns:
        None. Writes nested outputs under the configured folder.
    """

    print("Event Latency (nested)")
    env = NestedEnvironment()
    env._ns_print_branch_summary = False
    cable = Cable(env, 10)
    # Output options first.
    env.set_output_options(out_dir=str(NESTED_OUTPUT_FOLDER), gzip_trace=False)
    env.set_post_processing_options(
        gzip_trace=False,
        package_latest=True,
        print_outputs=False,
        autoplot=DEFAULT_AUTOPLOT,
        autoplot_example="event_latency",
        autoplot_state_field="state_item_count",
        autoplot_skip_no_branch=True,
    )
    # Outer settings.
    env.set_rng("independent")
    env.set_outer_seed(42)
    env.set_nested_triggering_objects(nested_id="cable_store")
    env.set_nesting_conditions({"on": "store_put", "frequency": 1})
    env.set_outer_stopping_condition(timeout=SIM_DURATION)
    # Inner settings.
    env.set_inner_repetitions(1)
    env.set_inner_stopping_condition(
        relative_time=20.0, triggering_customer_departs=True
    )
    env.process(sender(env, cable))
    env.process(receiver(env, cable))
    env.nested_run()


if __name__ == "__main__":
    main()


In [ ]:
# Run as a subprocess so the outer output is clean (inner branches fork).
!python event_latency_colab.py

## 3. Inspect the run

`OutputManager` reads the run folder and reports the triggering events, plots the outer trajectory, and exports the sample path.

In [ ]:
import glob, os
run = os.path.dirname(glob.glob("simpy_examples/event_latency/**/raw", recursive=True)[0])

from nestedsimpy import OutputManager
om = OutputManager(run)
print(f"{len(om.triggers())} triggering events; the outer path has {len(om.export_outer())} recorded events")

fig = om.visualize_outer()        # outer trajectory with triggering markers
fig.show()

om.export_outer("outer.csv")      # the outer sample path as a CSV
print("wrote outer.csv")